# ?? ComfyUI + AnimateDiff ??? Google Colab

## ???? ????? ??????? ??????? ?????????

### ?? ???????:
1. ? ?????? ?? GPU
2. ?? ??????? ComfyUI
3. ?? ????? ?????????
4. ?? ????? ???????
5. ?? ????? ??????
6. ?? ????? AI ?????????

---

## 1?? ?????? ?? GPU ?????? ?????????

In [ ]:
# ?????? ?? GPU
!nvidia-smi

import os
HOME = os.path.expanduser("~")
print(f"\n? Home directory: {HOME}")
print("? GPU ????!")

## 2?? ??????? ?????? ComfyUI

In [ ]:
# ???????? ?????? ???????
%cd {HOME}

# ??? ?????? ?????? ?? ???
!rm -rf ComfyUI

# ??????? ComfyUI
!git clone https://github.com/comfyanonymous/ComfyUI

print("\n? ?? ??????? ComfyUI ?????!")

## 3?? ????? ?????????

In [ ]:
# ???????? ????? ComfyUI
%cd {HOME}/ComfyUI

# ????? ?????????
!pip install -q -r requirements.txt

# ????? ?????? ??????
!pip install -q xformers==0.0.27.post2 --no-dependencies

print("\n? ?? ????? ???? ????????? ?????!")

## 4?? ????? ??????? ????????

? **??????? 5-10 ????? (~8 GB)**

In [ ]:
import os
from pathlib import Path

# ????? ?????? ????????
BASE_PATH = Path(HOME) / "ComfyUI"
checkpoints_dir = BASE_PATH / "models" / "checkpoints"
vae_dir = BASE_PATH / "models" / "vae"
animatediff_dir = BASE_PATH / "custom_nodes" / "ComfyUI-AnimateDiff-Evolved" / "models"

# ????? ????????
os.makedirs(checkpoints_dir, exist_ok=True)
os.makedirs(vae_dir, exist_ok=True)
os.makedirs(animatediff_dir, exist_ok=True)

print("? ?? ????? ????????")
print(f"?? Checkpoints: {checkpoints_dir}")
print(f"?? VAE: {vae_dir}")
print(f"?? AnimateDiff: {animatediff_dir}")

In [ ]:
# ????? Stable Diffusion v1.5 Checkpoint
import os

checkpoint_file = f"{checkpoints_dir}/v1-5-pruned-emaonly.safetensors"

if not os.path.exists(checkpoint_file):
    print("?? ????? Stable Diffusion v1.5 (4.27 GB)...")
    !wget -q --show-progress -O "{checkpoint_file}" \
        https://huggingface.co/runwayml/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.safetensors
    print("? ?? ????? Checkpoint")
else:
    print("? Checkpoint ????? ??????")

In [ ]:
# ????? VAE
vae_file = f"{vae_dir}/vae-ft-mse-840000-ema-pruned.safetensors"

if not os.path.exists(vae_file):
    print("?? ????? VAE (335 MB)...")
    !wget -q --show-progress -O "{vae_file}" \
        https://huggingface.co/stabilityai/sd-vae-ft-mse-original/resolve/main/vae-ft-mse-840000-ema-pruned.safetensors
    print("? ?? ????? VAE")
else:
    print("? VAE ????? ??????")

In [ ]:
# ????? ComfyUI-AnimateDiff-Evolved
%cd {HOME}/ComfyUI/custom_nodes

if not os.path.exists("ComfyUI-AnimateDiff-Evolved"):
    print("?? ??????? AnimateDiff Custom Node...")
    !git clone https://github.com/Kosinkadink/ComfyUI-AnimateDiff-Evolved.git
    print("? ?? ??????? AnimateDiff Node")
else:
    print("? AnimateDiff Node ????? ??????")

# ????? AnimateDiff Motion Module
%cd {HOME}/ComfyUI

animatediff_file = f"{animatediff_dir}/mm_sd_v15_v2.ckpt"

if not os.path.exists(animatediff_file):
    print("?? ????? AnimateDiff Motion Module (1.82 GB)...")
    !wget -q --show-progress -O "{animatediff_file}" \
        https://huggingface.co/guoyww/animatediff/resolve/main/mm_sd_v15_v2.ckpt
    print("? ?? ????? AnimateDiff Module")
else:
    print("? AnimateDiff Module ????? ??????")

In [ ]:
# ????? VideoHelperSuite ???????
%cd {HOME}/ComfyUI/custom_nodes

if not os.path.exists("ComfyUI-VideoHelperSuite"):
    print("?? ????? Video Helper Suite...")
    !git clone https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git
    %cd ComfyUI-VideoHelperSuite
    !pip install -q -r requirements.txt
    print("? ?? ????? Video Helper Suite")
else:
    print("? Video Helper Suite ????? ??????")

%cd {HOME}/ComfyUI

## 5?? ????? ???? ComfyUI

?? **???:** ????? ?????? ????????. ?????? ?????? ????? ?????? ???????.

### ?? ?????? ???????:
- ???? ??? ?????? ???? ???? ??? ????? ??????
- ?? ?????? **Cloudflare Tunnel** ?????? ?????

In [ ]:
# ????? ComfyUI ?? Cloudflare Tunnel ?????? ?? ??????
# ????? cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

print("? ?? ????? Cloudflare Tunnel")

In [ ]:
# ????? ComfyUI ???? Tunnel
import subprocess
import threading
import time
import re

%cd {HOME}/ComfyUI

def run_comfyui():
    """????? ComfyUI"""
    subprocess.run(['python', 'main.py', '--listen', '0.0.0.0', '--port', '8188'])

def run_tunnel():
    """????? Cloudflare Tunnel"""
    time.sleep(5)  # ?????? ComfyUI ?????
    process = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', 'http://localhost:8188'],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        universal_newlines=True
    )
    
    # ????? URL
    for line in process.stderr:
        if 'trycloudflare.com' in line:
            url = re.search(r'https://[^\s]+trycloudflare.com', line)
            if url:
                print(f"\n" + "="*60)
                print(f"?? ???? ?????? ???????:")
                print(f"?? {url.group(0)}")
                print(f"="*60 + "\n")
                break

# ????? ?? threads ??????
print("?? ???? ????? ComfyUI...")
print("? ????? ?????? ?????? ??? ???? ??????...\n")

comfyui_thread = threading.Thread(target=run_comfyui, daemon=True)
tunnel_thread = threading.Thread(target=run_tunnel, daemon=True)

comfyui_thread.start()
tunnel_thread.start()

# ????? ??? cell ????
print("\n? ComfyUI ???? ????!")
print("?? ?????? ?????? ????? ?????? ???????")
print("\n?? ?? ???? ??? ?????? - ???? ????!\n")

# ????? ???????? ????
try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    print("\n?? ?? ????? ??????")

## 6?? ????? AI ????????? (???????)

?????? ??? ??????? ???? ??????? ??? AI

In [ ]:
# ????? AI ????????? (????? Colab AI API)
try:
    import ipywidgets as widgets
    from IPython.display import display, HTML, Markdown, clear_output
    from google.colab import ai

    dropdown = widgets.Dropdown(
        options=[],
        layout={'width': 'auto'}
    )

    def update_model_list(new_options):
        dropdown.options = new_options
    
    try:
        update_model_list(ai.list_models())
    except:
        update_model_list(['gemini-pro'])

    text_input = widgets.Textarea(
        placeholder='?????? ?? ??? ?? ComfyUI ?? AnimateDiff...',
        layout={'width': '100%', 'height': '100px'},
    )

    button = widgets.Button(
        description='?????',
        disabled=False,
        tooltip='???? ???????',
        icon='check'
    )

    output_area = widgets.Output(
        layout={'width': 'auto', 'max_height': '300px','overflow_y': 'scroll'}
    )

    def on_button_clicked(b):
        with output_area:
            output_area.clear_output(wait=False)
            accumulated_content = ""
            try:
                for new_chunk in ai.generate_text(prompt=text_input.value, model_name=dropdown.value, stream=True):
                    if new_chunk is None:
                        continue
                    accumulated_content += new_chunk
                    clear_output(wait=True)
                    display(Markdown(accumulated_content))
            except Exception as e:
                display(Markdown(f"**???:** {str(e)}\n\n*?? ????? ??? ????? Google AI API*"))

    button.on_click(on_button_clicked)
    vbox = widgets.VBox([dropdown, text_input, button, output_area])

    display(HTML("""
    <div style='background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); 
                padding: 20px; border-radius: 10px; margin-bottom: 20px;'>
        <h2 style='color: white; margin: 0;'>?? ????? AI ????????</h2>
    </div>
    """))
    display(vbox)

except ImportError:
    print("?? ????? AI ??? ????? ?? ??? ??????")

---

## ?? ???? ?????????

### ?? ????? ????? ?????:

1. **???? ???????** ?? ?????? ?????
2. **??? Nodes** ?????? ?? ?????? Workflow ????
3. **???? Prompts:**
   - Positive: `stunning cinematic landscape, mountains, sunrise, 8k`
   - Negative: `low quality, blurry, watermark`
4. **???? ?????????:**
   - Frames: 16
   - Steps: 20
   - CFG: 7.5
5. **???? Queue Prompt**
6. **???? ???????** ?? ???? Output

### ?? ????? ???????:

```python
from google.colab import files
import glob

# ??? ?????????? ???????
videos = glob.glob('/root/ComfyUI/output/*.mp4')
print("?????????? ???????:")
for v in videos:
    print(f"  - {v}")

# ????? ??? ?????
if videos:
    latest = max(videos, key=os.path.getctime)
    print(f"\n?????: {latest}")
    files.download(latest)
```

### ?? ?????:

- ?????? **T4 GPU** (????? ??? Colab)
- ??? ??? ??? Frames ??? ???? ???????
- ???? ???? ??? ?????? ??????
- ?????????? ????? ?? `/root/ComfyUI/output/`

---

## ?? ?? ???????

### ? Out of Memory:
- ??? batch_size ?? 16 ??? 8
- ??? resolution ?? 512 ??? 384

### ? ?????? ?? ????:
- ???? ?? ????? ???? "????? ComfyUI"
- ????? 1-2 ????? ?????? ??? ??????

### ? ??????? ??????:
- ??? ????? ????? ???????
- ???? ?? ????????

---

## ? ????? ?????

- [ComfyUI Documentation](https://github.com/comfyanonymous/ComfyUI)
- [AnimateDiff Guide](https://github.com/Kosinkadink/ComfyUI-AnimateDiff-Evolved)
- [Workflow Examples](https://comfyworkflows.com/)

---

<div style='text-align: center; padding: 20px; background: #f0f0f0; border-radius: 10px;'>
  <h3>?? ???????? ?? ????? ????? ????? ?????!</h3>
  <p>???? ?? ?? ??????? ??????</p>
</div>

## ?? ????? ?????????? ????????

In [ ]:
# ????? ???? ?????????? ?? ???? Output
from google.colab import files
import glob
import os

output_dir = f"{HOME}/ComfyUI/output"

# ????? ?? ???? ??????????
videos = glob.glob(f"{output_dir}/*.mp4") + glob.glob(f"{output_dir}/*.gif")

if videos:
    print(f"?? ?? ?????? ??? {len(videos)} ?????:\n")
    for i, video in enumerate(videos, 1):
        print(f"  {i}. {os.path.basename(video)}")
    
    # ????? ??? ?????
    latest = max(videos, key=os.path.getctime)
    print(f"\n?? ????? ??? ?????: {os.path.basename(latest)}")
    files.download(latest)
else:
    print("? ?? ??? ?????? ??? ????????")
    print("?? ?? ?????? ????? ?? ????? ComfyUI ?????")